**Análisis comparativo de arquitecturas Transformer para visión: ViT, DeiT, Swin-T y ResNet-50**

Evaluación sobre Imagenette (300 imágenes con etiquetas): accuracy, confianza, calibración, correlación y mapas de atención

 Configuración del entorno. Importación de librerías, selección de GPU y creación de la carpeta de salida

In [ ]:
!pip install -q transformers timm Pillow matplotlib seaborn numpy torch torchvision

import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import models, transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification

# Semilla fija para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Dispositivo: GPU si está disponible, si no CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

 Descarga del dataset. Se descarga Imagenette (subconjunto de 10 clases de ImageNet) directamente de fast.ai

In [ ]:
import os

# Descarga directa de fast.ai (sin token de Kaggle).
# Imagenette = subconjunto de 10 clases de ImageNet.
DATA_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"

import urllib.request, tarfile

DATA_ROOT = Path("imagenette2-320")
if not DATA_ROOT.exists():
    print("Descargando Imagenette...")
    urllib.request.urlretrieve(DATA_URL, "imagenette.tgz")
    with tarfile.open("imagenette.tgz") as t:
        t.extractall(".")
    print("Descarga y extracción completas.")
else:
    print("Imagenette ya está descargado.")

# Usamos la carpeta de validación
VAL_DIR = DATA_ROOT / "val"

Preparación de etiquetas. Se construye la correspondencia entre el código synset de cada clase y su índice (0-999) en ImageNet

In [ ]:
# Las 10 clases de Imagenette y su synset de ImageNet
IMAGENETTE_SYNSETS = {
    "n01440764": "tench",            # un pez
    "n02102040": "English springer", # un perro
    "n02979186": "cassette player",
    "n03000684": "chain saw",
    "n03028079": "church",
    "n03394916": "French horn",
    "n03417042": "garbage truck",
    "n03425413": "gas pump",
    "n03445777": "golf ball",
    "n03888257": "parachute",
}

# Nombres de las 1000 clases de ImageNet, en el orden 0-999 estándar.
# torchvision los trae en los metadatos de los pesos de ResNet.
imagenet_labels = models.ResNet50_Weights.IMAGENET1K_V1.meta["categories"]

# Construimos: synset -> índice 0-999
# buscando el nombre de cada synset dentro de la lista de 1000 nombres.
synset_to_idx = {}
for synset, name in IMAGENETTE_SYNSETS.items():
    idx = imagenet_labels.index(name)
    synset_to_idx[synset] = idx

print("Mapeo synset -> índice ImageNet:")
for s, i in synset_to_idx.items():
    print(f"  {s} ({IMAGENETTE_SYNSETS[s]}) -> clase {i}")

Carga de imágenes con etiqueta. Se cargan 30 imágenes por clase (300 en total) junto con su etiqueta verdadera





In [ ]:
N_PER_CLASS = 30

images = []
true_labels = []   # el índice 0-999 correcto de cada imagen

for synset, idx in synset_to_idx.items():
    class_dir = VAL_DIR / synset
    files = sorted(class_dir.glob("*.JPEG"))[:N_PER_CLASS]
    for f in files:
        img = Image.open(f).convert("RGB")
        images.append(img)
        true_labels.append(idx)

print(f"\nTotal imágenes cargadas: {len(images)}")
print(f"Etiquetas verdaderas disponibles: {len(true_labels)}")

Carga de los cuatro modelos preentrenados en modo evaluación: ViT-B/16, DeiT-B, Swin-T (Hugging Face) y ResNet-50 (torchvision)

In [ ]:
# ---------- ViT-B/16 (Google, 2020) ----------
VIT_HF = "google/vit-base-patch16-224"
vit_proc  = AutoImageProcessor.from_pretrained(VIT_HF)
vit_model = AutoModelForImageClassification.from_pretrained(
    VIT_HF, output_attentions=True
).to(DEVICE).eval()
print("  ✓ ViT-B/16 cargado")

# ---------- DeiT-B (Meta, 2021) ----------
DEIT_HF = "facebook/deit-base-distilled-patch16-224"
deit_proc  = AutoImageProcessor.from_pretrained(DEIT_HF)
deit_model = AutoModelForImageClassification.from_pretrained(
    DEIT_HF, output_attentions=True
).to(DEVICE).eval()
print("  ✓ DeiT-B cargado")

# ---------- Swin-T (Microsoft, 2021) ----------
SWIN_HF = "microsoft/swin-tiny-patch4-window7-224"
swin_proc  = AutoImageProcessor.from_pretrained(SWIN_HF)
swin_model = AutoModelForImageClassification.from_pretrained(
    SWIN_HF
).to(DEVICE).eval()
print("  ✓ Swin-T cargado")

# ---------- ResNet-50 (He et al., 2015) ----------
resnet_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
resnet_model = resnet_model.to(DEVICE).eval()
resnet_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
print("  ✓ ResNet-50 cargado")

# Diccionario unificado para iterar fácilmente
MODELS = {
    "ViT-B/16":  {"model": vit_model,    "proc": vit_proc,    "type": "hf"},
    "DeiT-B":    {"model": deit_model,   "proc": deit_proc,   "type": "hf"},
    "Swin-T":    {"model": swin_model,   "proc": swin_proc,   "type": "hf"},
    "ResNet-50": {"model": resnet_model, "proc": resnet_transform, "type": "tv"},
}
print("\nTodos los modelos listos.")

Inferencia. Cada imagen se pasa por los cuatro modelos, obteniendo confianza, predicción y aciertos Top-1 y Top-5

In [ ]:
def run_inference_hf(model, processor, img, device):
    """Inferencia para modelos Hugging Face (ViT, DeiT, Swin)."""
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs)
    probs     = F.softmax(out.logits, dim=-1)[0]
    top1_conf = probs.max().item() * 100
    top1_id   = probs.argmax().item()
    attentions = out.attentions if hasattr(out, "attentions") else None
    return top1_conf, top1_id, probs, attentions

def run_inference_tv(model, transform, img, device):
    """Inferencia para modelos torchvision (ResNet)."""
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
    probs     = F.softmax(logits, dim=-1)[0]
    top1_conf = probs.max().item() * 100
    top1_id   = probs.argmax().item()
    return top1_conf, top1_id, probs, None

results = {name: {"confs": [], "pred_ids": [], "correct_top1": [], "correct_top5": []}
           for name in MODELS}
raw_attentions = {name: [] for name in ["ViT-B/16", "DeiT-B"]}

print("Ejecutando inferencia...")
for img_idx, img in enumerate(images):
    true_id = true_labels[img_idx]   # la clase CORRECTA de esta imagen

    for name, cfg in MODELS.items():
        if cfg["type"] == "hf":
            conf, pred_id, probs, attns = run_inference_hf(
                cfg["model"], cfg["proc"], img, DEVICE
            )
            if name in raw_attentions:
                raw_attentions[name].append(attns)
        else:
            conf, pred_id, probs, _ = run_inference_tv(
                cfg["model"], cfg["proc"], img, DEVICE
            )

        # Top-5: ¿está la clase correcta entre las 5 más probables?
        top5_ids = probs.topk(5).indices.tolist()

        results[name]["confs"].append(conf)
        results[name]["pred_ids"].append(pred_id)
        results[name]["correct_top1"].append(pred_id == true_id)
        results[name]["correct_top5"].append(true_id in top5_ids)

    if (img_idx + 1) % 50 == 0:
        print(f"  {img_idx+1}/{len(images)} imágenes procesadas")

print("\nInferencia completa.")

 Estadísticas. Cálculo de accuracy, confianza media y calibración (confianza en aciertos frente a fallos)

In [ ]:
import numpy as np

print(f"{'Modelo':<12} {'Acc@1':>7} {'Acc@5':>7} {'Conf.media':>11} {'Conf.std':>9}")
print("─" * 50)

stats = {}
for name, res in results.items():
    confs        = np.array(res["confs"])
    acc_top1     = np.mean(res["correct_top1"]) * 100
    acc_top5     = np.mean(res["correct_top5"]) * 100

    # Confianza separada en aciertos vs fallos (clave para calibración)
    correct_mask = np.array(res["correct_top1"])
    conf_acierto = confs[correct_mask].mean()  if correct_mask.any()      else 0
    conf_fallo   = confs[~correct_mask].mean() if (~correct_mask).any()   else 0

    stats[name] = {
        "acc_top1": acc_top1,
        "acc_top5": acc_top5,
        "conf_mean": confs.mean(),
        "conf_std": confs.std(),
        "conf_acierto": conf_acierto,
        "conf_fallo": conf_fallo,
        "confs": confs,
    }

    print(f"{name:<12} {acc_top1:>6.1f}% {acc_top5:>6.1f}% "
          f"{confs.mean():>10.1f}% {confs.std():>8.1f}%")

print("─" * 50)

# Tabla extra: el análisis de calibración
print("\nConfianza media según acierto/fallo:")
print(f"{'Modelo':<12} {'en aciertos':>12} {'en fallos':>11}")
print("─" * 37)
for name in MODELS:
    print(f"{name:<12} {stats[name]['conf_acierto']:>11.1f}% "
          f"{stats[name]['conf_fallo']:>10.1f}%")

Visualizaciones: boxplot de confianza, barras de accuracy, matriz de correlación y gráfico de calibración

In [ ]:
import seaborn as sns
MODEL_NAMES = list(MODELS.keys())
COLORS_PLOT = ["#378ADD", "#1D9E75", "#D85A30", "#888780"]

# ---------- Boxplot de confianza ----------
fig, ax = plt.subplots(figsize=(8, 5))
data_box = [stats[n]["confs"] for n in MODEL_NAMES]
bp = ax.boxplot(data_box, patch_artist=True, widths=0.45,
                medianprops={"color": "white", "linewidth": 2})
for patch, color in zip(bp["boxes"], COLORS_PLOT):
    patch.set_facecolor(color + "99")
    patch.set_edgecolor(color)
ax.set_xticks(range(1, len(MODEL_NAMES) + 1))
ax.set_xticklabels(MODEL_NAMES)
ax.set_ylabel("Confianza Top-1 (%)")
ax.set_title(f"Distribución de confianza — {len(images)} imágenes")
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "boxplot_confianza.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- NUEVO: barras de accuracy Top-1 y Top-5 ----------
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(MODEL_NAMES))
w = 0.35
acc1 = [stats[n]["acc_top1"] for n in MODEL_NAMES]
acc5 = [stats[n]["acc_top5"] for n in MODEL_NAMES]
ax.bar(x - w/2, acc1, w, label="Top-1", color="#378ADD")
ax.bar(x + w/2, acc5, w, label="Top-5", color="#1D9E75")
for i, (a1, a5) in enumerate(zip(acc1, acc5)):
    ax.text(i - w/2, a1 + 1, f"{a1:.1f}", ha="center", fontsize=8)
    ax.text(i + w/2, a5 + 1, f"{a5:.1f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES)
ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 105)
ax.set_title("Accuracy Top-1 vs Top-5 por modelo")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "barras_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

conf_matrix = np.array([stats[n]["confs"] for n in MODEL_NAMES])
corr_matrix = np.corrcoef(conf_matrix)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr_matrix, annot=True, fmt=".2f",
            xticklabels=MODEL_NAMES, yticklabels=MODEL_NAMES,
            cmap="YlGnBu", vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Correlación de Pearson entre confianzas")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "heatmap_correlacion.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- NUEVO: gráfico de calibración (confianza en aciertos vs fallos) ----------
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(MODEL_NAMES))
w = 0.35
conf_ok   = [stats[n]["conf_acierto"] for n in MODEL_NAMES]
conf_bad  = [stats[n]["conf_fallo"]   for n in MODEL_NAMES]
ax.bar(x - w/2, conf_ok,  w, label="Confianza cuando ACIERTA", color="#1D9E75")
ax.bar(x + w/2, conf_bad, w, label="Confianza cuando FALLA",   color="#D85A30")
for i, (c_ok, c_bad) in enumerate(zip(conf_ok, conf_bad)):
    ax.text(i - w/2, c_ok + 1,  f"{c_ok:.1f}", ha="center", fontsize=8)
    ax.text(i + w/2, c_bad + 1, f"{c_bad:.1f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES)
ax.set_ylabel("Confianza media (%)"); ax.set_ylim(0, 105)
ax.set_title("Calibración: confianza media en aciertos vs fallos")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "calibracion.png", dpi=150, bbox_inches="tight")
plt.show()

Selección de casos representativos: una imagen fácil, una de desacuerdo y una difícil

In [ ]:
n_imgs = len(images)

def score_caso_A(i):
    """Caso A: todos aciertan Y con alta confianza (caso fácil ideal)."""
    todos_aciertan = all(results[m]["correct_top1"][i] for m in MODEL_NAMES)
    if not todos_aciertan:
        return -1
    return min(results[m]["confs"][i] for m in MODEL_NAMES)

def score_caso_B(i):
    """Caso B: desacuerdo entre modelos (unos aciertan, otros fallan)."""
    aciertos = [results[m]["correct_top1"][i] for m in MODEL_NAMES]
    # máxima discrepancia = mezcla de aciertos y fallos
    n_aciertos = sum(aciertos)
    if n_aciertos == 0 or n_aciertos == 4:
        return -1            # ni todos aciertan ni todos fallan
    return min(n_aciertos, 4 - n_aciertos)   # más cerca de 2-2 = más discrepancia

def score_caso_C(i):
    """Caso C: todos fallan (caso difícil)."""
    todos_fallan = all(not results[m]["correct_top1"][i] for m in MODEL_NAMES)
    if not todos_fallan:
        return -1
    # entre los que todos fallan, el de confianza media más alta (fallo "seguro")
    return np.mean([results[m]["confs"][i] for m in MODEL_NAMES])

idx_A = max(range(n_imgs), key=score_caso_A)
idx_B = max(range(n_imgs), key=score_caso_B)
idx_C = max(range(n_imgs), key=score_caso_C)

repr_indices = {
    "A — todos aciertan (fácil)":      idx_A,
    "B — desacuerdo entre modelos":    idx_B,
    "C — todos fallan (difícil)":      idx_C,
}

print("Imágenes representativas seleccionadas:")
for label, idx in repr_indices.items():
    true_name = imagenet_labels[true_labels[idx]]
    print(f"\n  {label}  →  índice {idx}  (clase real: {true_name})")
    for m in MODEL_NAMES:
        pred_name = imagenet_labels[results[m]["pred_ids"][idx]]
        ok = "✓" if results[m]["correct_top1"][idx] else "✗"
        print(f"      {m:<12}: {pred_name:<20} {results[m]['confs'][idx]:>5.1f}% {ok}")

Extracción de mapas de atención de ViT y DeiT para las tres imágenes representativas

In [ ]:
def extract_attention_map(model, processor, img, device):
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs, output_attentions=True)

    # out.attentions: una entrada por capa. Cogemos la ÚLTIMA capa.
    last_attn = out.attentions[-1][0]   # forma: (n_cabezas, seq, seq)
    avg_attn  = last_attn.mean(dim=0)   # promediamos las cabezas -> (seq, seq)

    # Detectar cuántos tokens especiales hay (ViT=1 [CLS], DeiT=2 [CLS]+[dist])
    seq_len   = avg_attn.shape[0]
    n_patches = 196                     # 224/16 = 14, y 14x14 = 196 parches
    n_special = seq_len - n_patches     # ViT -> 1 | DeiT -> 2

    # Atención DESDE el token [CLS] HACIA todos los parches
    cls_attn  = avg_attn[0, n_special:]  # forma: (196,)

    # Reorganizar los 196 valores en una rejilla 14x14
    grid_size = int(n_patches ** 0.5)   # 14
    attn_map  = cls_attn.reshape(grid_size, grid_size).cpu().numpy()

    # Redimensionar de 14x14 a 224x224 para superponer sobre la imagen
    attn_pil  = Image.fromarray(attn_map).resize((224, 224), Image.BILINEAR)
    attn_arr  = np.array(attn_pil)
    # Normalizar a [0,1] para visualizar
    attn_arr  = (attn_arr - attn_arr.min()) / (attn_arr.max() - attn_arr.min() + 1e-8)
    return attn_arr

# Calcular mapas para las 3 imágenes representativas, solo ViT y DeiT
attention_maps = {}
letter_map = {
    "A — todos aciertan (fácil)":   "A",
    "B — desacuerdo entre modelos": "B",
    "C — todos fallan (difícil)":   "C",
}

for label, idx in repr_indices.items():
    letter = letter_map[label]
    attention_maps[letter] = {}
    img = images[idx]
    for mname in ["ViT-B/16", "DeiT-B"]:
        cfg = MODELS[mname]
        attn = extract_attention_map(cfg["model"], cfg["proc"], img, DEVICE)
        attention_maps[letter][mname] = attn
        print(f"  ✓ Mapa de atención: imagen {letter} — {mname}")

print("\nExtracción de mapas de atención completa.")

Figuras finales comparativas: para cada caso, imagen + predicción + mapa de atención + Top-3 de los cuatro modelos

In [ ]:
imagenet_labels_full = models.ResNet50_Weights.IMAGENET1K_V1.meta["categories"]

def top3_info(model_name, img_idx):
    """Devuelve [(nombre_clase, confianza%), ...] del Top-3 para una imagen."""
    cfg = MODELS[model_name]
    img = images[img_idx]
    if cfg["type"] == "hf":
        inputs = cfg["proc"](images=img, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            logits = cfg["model"](**inputs).logits
        probs = F.softmax(logits, dim=-1)[0]
    else:
        tensor = cfg["proc"](img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = cfg["model"](tensor)
        probs = F.softmax(logits, dim=-1)[0]
    top3_vals, top3_ids = probs.topk(3)
    return [(imagenet_labels_full[cid.item()], val.item() * 100)
            for cid, val in zip(top3_ids, top3_vals)]

BAR_COLORS = ["#378ADD", "#1D9E75", "#D85A30", "#888780"]

def plot_representative_image(letter, idx, label_title):
    fig, axes = plt.subplots(4, 3, figsize=(13, 14),
                             gridspec_kw={"width_ratios": [1.3, 1.3, 1]})
    true_name = imagenet_labels_full[true_labels[idx]]
    fig.suptitle(f"Imagen {letter} — {label_title}\nClase real: {true_name}",
                 fontsize=13, y=1.01, fontweight="bold")
    img_arr = np.array(images[idx])
    has_attn = {"ViT-B/16": True, "DeiT-B": True, "Swin-T": False, "ResNet-50": False}

    for row, (mname, color) in enumerate(zip(MODEL_NAMES, BAR_COLORS)):
        ax_img, ax_attn, ax_bars = axes[row]

        # Columna 0: imagen + predicción, con ✓/✗ según si acertó
        ax_img.imshow(img_arr)
        top3 = top3_info(mname, idx)
        acerto = results[mname]["correct_top1"][idx]
        marca = "✓" if acerto else "✗"
        ax_img.set_title(f"{mname}\nTop-1: {top3[0][0]} {marca}\n{top3[0][1]:.1f}%",
                         fontsize=9, loc="left", pad=3,
                         color=("green" if acerto else "red"))
        ax_img.axis("off")

        # Columna 1: mapa de atención (solo ViT y DeiT)
        if has_attn[mname]:
            attn = attention_maps[letter][mname]
            ax_attn.imshow(img_arr)
            ax_attn.imshow(attn, cmap=plt.cm.inferno, alpha=0.55)
            ax_attn.set_title("Mapa de atención\n(capa final, CLS)", fontsize=8)
        else:
            ax_attn.imshow(img_arr, alpha=0.25)
            ax_attn.text(0.5, 0.5, f"Atención no\ndisponible\npara {mname}",
                         ha="center", va="center", transform=ax_attn.transAxes,
                         fontsize=9, color="gray")
        ax_attn.axis("off")

        # Columna 2: barras Top-3
        clases = [t[0][:20] for t in top3]
        confs  = [t[1] for t in top3]
        y_pos  = [2, 1, 0]
        bars = ax_bars.barh(y_pos, confs, color=color + "aa",
                            edgecolor=color, height=0.55)
        for bar, conf in zip(bars, confs):
            ax_bars.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                         f"{conf:.1f}%", va="center", fontsize=8)
        ax_bars.set_yticks(y_pos); ax_bars.set_yticklabels(clases, fontsize=8)
        ax_bars.set_xlim(0, 110); ax_bars.set_title("Top-3", fontsize=9)
        ax_bars.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    fname = OUTPUT_DIR / f"figura_imagen_{letter}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Figura imagen {letter} guardada: {fname}")

# Generar las 3 figuras
for label, idx in repr_indices.items():
    letter = letter_map[label]
    plot_representative_image(letter, idx, label)


